# Improving the vocoder, one change at a time

At the end of notebook 03, 20,000 GAN steps on a laptop gave us a vocoder that was semi-tolerable. Three changes
should make it better:

1. **Train on all of LJSpeech**, not the first 500 clips.
2. **Decay the learning rate** over the run.
3. **Widen the generator** from 128 to 256 channels.

We want to know how much each one buys, so we'll build a ladder. Each rung keeps everything from the rung below and
adds exactly one change, with the same budget: 1,500 steps of mel loss then 20,000 GAN steps. Each rung saves a
checkpoint, so rerunning the notebook reloads rather than retrains.

| Rung | Change | Training time on an M3 Pro |
|---|---|---|
| 0 | notebook 03's model | already trained |
| 1 | + all clips | ~2 h |
| 2 | + learning rate decay | ~2 h |
| 3 | + 256 channels | ~2.5 h |

The generator, discriminator and training loops are exported from notebook 03 to `text_to_speech.vocoder`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from IPython.display import Audio, display

from text_to_speech.audio import SR, log_mel
from text_to_speech.core import DATA, default_device
from text_to_speech.data import metadata
from text_to_speech.vocoder import Discriminator, Generator, pretrained_hifigan, train_gan, train_mel

device = default_device()
checkpoints = DATA / "checkpoints"
mel_steps, gan_steps = 1_500, 20_000

## A fair test

Notebook 03 had a flaw. It trained on the first 500 files in sorted order and tested on `LJ001-0001`, which is the
very first of them. After 20,000 steps the model had seen those 55 minutes of audio about 36 times, so some of what we
heard may have been memorised.

From here on we hold out the last chapter of the book, the 278 clips named `LJ050-*`. No rung ever trains on them.

In [ ]:
df = metadata()
is_test = df["id"].str.startswith("LJ050")
train_clips = [sf.read(p, dtype="int16")[0] for p in df[~is_test]["path"]]
test_clips = [torch.tensor(sf.read(p, dtype="float32")[0]) for p in df[is_test]["path"][:30]]

print(f"train: {len(train_clips):,} clips, {sum(len(c) for c in train_clips) / SR / 3600:.1f} hours")
print(f"test:  {len(test_clips)} clips (of {is_test.sum()} held out)")

## A better metric

Notebook 03's other lesson was that mel error is a poor judge: Griffin-Lim beat everything on it and sounded the
worst. So we'll also use **UTMOS**, a 100M parameter network trained to predict the 1 to 5 score human listeners gave
to thousands of synthetic speech clips in the VoiceMOS 2022 challenge. It's what Piper logs during training. It isn't
a human, but it ranks things the way ears do: Griffin-Lim scores about 1.8, our 20k-step model about 2.5, the
pretrained HiFi-GAN about 4.2, and real LJSpeech about 4.4.

`evaluate` averages three numbers over the 30 test clips:

- **MOS**: UTMOS's predicted score. Higher is better.
- **mel error**: mean absolute difference of log mels, as in notebook 03. Lower is better, with the caveat above.
- **loudness**: RMS of the output divided by RMS of the real clip. Our models have been coming out quiet, so 1.0 is
  the target.

In [ ]:
utmos = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True).eval()


def evaluate(generator):
    scores = []
    for real in test_clips:
        mel = log_mel(real[None].to(device))
        with torch.no_grad():
            fake = generator(mel).flatten()
            n = mel.shape[-1]
            mos = utmos(fake[None].cpu(), SR).item()
            error = (log_mel(fake[None])[..., :n] - mel[..., :n]).abs().mean().item()
        loudness = (fake.pow(2).mean().sqrt() / real.pow(2).mean().sqrt()).item()
        scores.append((mos, error, loudness))
    return dict(zip(["MOS", "mel error", "loudness"], np.mean(scores, 0).round(3).tolist()))


def listen(generator, clips=(0, 1)):
    for i in clips:
        with torch.no_grad():
            display(Audio(generator(log_mel(test_clips[i][None].to(device))).flatten().cpu(), rate=SR))

First, the bounds. Real audio gives the ceiling for MOS. The pretrained HiFi-GAN from notebook 03 is what 2.5 million
steps on a V100 buys. Take its score with some salt, though: it was almost certainly trained on all of LJSpeech,
test clips included.

In [ ]:
real_mos = np.mean([utmos(c[None], SR).item() for c in test_clips])
print(f"real audio MOS: {real_mos:.3f}")

pretrained = pretrained_hifigan().to(device)
results = {"pretrained (14M, 2.5M steps)": evaluate(pretrained)}
results

In [ ]:
listen(pretrained)

## The ladder

`run` trains one rung, or loads it if its checkpoint already exists, then scores it and plays the first two test
clips. Seeding inside `run` means each rung gets the same random batches in the same order, whichever order you run
the cells in, so the only difference between rungs is the change we made.

In [ ]:
def run(name, clips, channels=128, final_lr=None):
    path = checkpoints / f"{name}.pt"
    generator = Generator(channels=channels).to(device)
    if path.exists():
        generator.load_state_dict(torch.load(path, map_location=device))
    else:
        torch.manual_seed(0)
        np.random.seed(0)
        train_mel(generator, clips, mel_steps)
        history = train_gan(generator, Discriminator().to(device), clips, gan_steps, final_lr=final_lr)
        torch.save(generator.state_dict(), path)
        np.save(checkpoints / f"{name}_history.npy", history)
    generator.eval()
    results[name] = evaluate(generator)
    print(results[name])
    listen(generator)
    return generator

### Rung 0: notebook 03's model

128 channels, the first 500 clips, a constant learning rate. This loads the checkpoint notebook 03 saved. If it isn't
there, `run` retrains it from scratch with the same recipe.

In [ ]:
rung0 = run("03_generator_gan", train_clips[:500])

### Rung 1: all the clips

Rung 0 saw 55 minutes of speech. Now it's 23 hours, 26 times more. In 20,000 steps of 16 segments of 0.37 seconds, the
model hears about 33 hours of audio, so it now sees each clip about 1.4 times instead of 36.

What to expect: less memorisation, so better on held-out clips. What not to expect: a big jump. A vocoder learns
something local, "how does this patch of mel turn into this patch of wave", and 55 minutes of one voice already
contains most of the patches there are. The places to listen are sounds that are rare in a short sample: breaths,
"sh", "z", unusual pitches.

In [ ]:
rung1 = run("04_all_clips", train_clips)

### Rung 2: learning rate decay

Early in training, big steps help the model find roughly the right weights quickly. Late in training they hurt,
because it keeps overshooting the fine detail. In a GAN it's worse: two networks are chasing each other, and a high
learning rate keeps both jittering around the equilibrium rather than settling into it. That jitter is audible as
roughness.

HiFi-GAN multiplies the learning rate by 0.999 after every epoch. Over the paper's 2.5 million steps that's about 3,000
epochs, taking it from 2e-4 to about 1e-5. But over our 20,000 steps it's about 25 epochs, and 0.999 to the 25th power
is 0.975, which would change nothing. So we keep the paper's start and end points and squeeze the same exponential
decay into our run: `train_gan` multiplies by a constant each step, chosen so the rate lands on `final_lr` at the end.

In [ ]:
rung2 = run("04_lr_decay", train_clips, final_lr=1e-5)

### Rung 3: a wider generator

Doubling the channels at every layer quadruples most of the weights: 0.93M parameters becomes 3.6M. The step only gets
about 30% slower, because on a laptop GPU the discriminator and the mel spectrograms are a big part of each step and
they haven't changed.

What to expect: capacity is what lets a vocoder render fine harmonic detail and clean noise at the same time, so this
is the change most likely to reduce the metallic edge on vowels. For our on-device goal, 3.6M parameters is still
tiny. Piper's whole model, text encoder and vocoder included, is around 20M.

In [ ]:
rung3 = run("04_wide", train_clips, channels=256, final_lr=1e-5)

## Results

In [ ]:
table = pd.DataFrame(results).T
table["MOS"].plot.barh(figsize=(8, 3), xlim=(1, 5), title="Predicted MOS on held-out clips")
plt.axvline(real_mos, color="k", ls="--", lw=1, label="real audio")
plt.legend()
plt.show()
table

And the training curves. Rung 0 has none, since notebook 03 didn't save one. These are mel losses on training
batches, smoothed over 500 steps. They show how each change affected optimisation, not how good the result sounds.

In [ ]:
plt.figure(figsize=(12, 3))
for name in ["04_all_clips", "04_lr_decay", "04_wide"]:
    path = checkpoints / f"{name}_history.npy"
    if path.exists():
        mel_loss = np.load(path)[:, 0]
        plt.plot(np.convolve(mel_loss, np.ones(500) / 500, mode="valid"), label=name)
plt.xlabel("GAN step")
plt.ylabel("mel loss")
plt.legend()
plt.show()

Finally, one held-out clip through every rung in order, so you can hear each change in turn.

In [ ]:
for name, g in [
    ("rung 0", rung0),
    ("rung 1: + all clips", rung1),
    ("rung 2: + lr decay", rung2),
    ("rung 3: + 256 channels", rung3),
    ("pretrained", pretrained),
]:
    print(name)
    listen(g, clips=(2,))
print("real")
display(Audio(test_clips[2], rate=SR))

## Things to try

- The ladder measures each change *given the ones before it*. Try the rungs in a different order, for example decay
  without all clips. Do the gains add up, or does one change mostly do the same job as another?
- Keep rung 3's recipe and double `gan_steps`. Is more time or more width the better buy?
- Widen the discriminator instead of the generator. The discriminator is thrown away after training, so on a phone
  its size is free.
- Listen to where UTMOS and your ears disagree. It's a model too, trained on other people's systems, and it has blind
  spots.